In [1]:
import sys, math
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

from qne.cascade.key import key_from_sifted_json
from qne.cascade.finite_key import finite_key_output_length, asymptotic_key_length, cascade_leakage, h, v
from randextract import ToeplitzHashing

print("Setup complete")

Setup complete


In [2]:
import json as _json

GEN_ALICE = PROJECT_DIR / "results" / "fabric_alice_sifted_bits_genonly.json"
GEN_BOB = PROJECT_DIR / "results" / "fabric_bob_sifted_bits_genonly.json"
META_PATH = PROJECT_DIR / "results" / "fabric_key0_pe_meta.json"

if not (GEN_ALICE.exists() and GEN_BOB.exists() and META_PATH.exists()):
    raise FileNotFoundError(
        "PE-split generation-only key files not found. Run "
        "10_sdc_real.ipynb's key-loading cell first -- it performs the "
        "PE split on the raw fabric_*_sifted_bits.json files and saves "
        "the generation-only bits + metadata used here, so every notebook "
        "analyzes the SAME split of key0 rather than each deriving its own."
    )

alice_key, alice_indices = key_from_sifted_json(str(GEN_ALICE), "alice_bits")
bob_key, bob_indices = key_from_sifted_json(str(GEN_BOB), "bob_bits")
assert alice_indices == bob_indices

meta = _json.loads(META_PATH.read_text())
k_pe, real_qber = meta["k"], meta["qber"]
print(f"Loaded {alice_key.get_nr_bits()} generation bits, k={k_pe} PE sample, "
      f"real_qber (from disjoint PE sample) = {real_qber:.4f}")


Loaded 3488 generation bits, k=388 PE sample, real_qber (from disjoint PE sample) = 0.0077


In [3]:
key_pairs_df = pd.read_csv(str(PROJECT_DIR / "results" / "key_pairs_metadata.csv"))
print(key_pairs_df)


   index  m_sifted  k_pe  n_bits      qber  \
0      0      3832   384    3448  0.013021   
1      1      3826   383    3443  0.007833   
2      2      3787   379    3408  0.018470   
3      3      3773   378    3395  0.018519   
4      4      3840   384    3456  0.010417   

                                          alice_path  \
0  /home/fabric/work/qkd-dependability/results/al...   
1  /home/fabric/work/qkd-dependability/results/al...   
2  /home/fabric/work/qkd-dependability/results/al...   
3  /home/fabric/work/qkd-dependability/results/al...   
4  /home/fabric/work/qkd-dependability/results/al...   

                                            bob_path  \
0  /home/fabric/work/qkd-dependability/results/bo...   
1  /home/fabric/work/qkd-dependability/results/bo...   
2  /home/fabric/work/qkd-dependability/results/bo...   
3  /home/fabric/work/qkd-dependability/results/bo...   
4  /home/fabric/work/qkd-dependability/results/bo...   

                                      alice_raw_p

In [4]:
from qne.cascade.finite_key import finite_key_output_length, asymptotic_key_length
from randextract import ToeplitzHashing

n = alice_key.get_nr_bits()
k = k_pe
Q = real_qber

out_len_finite, r, t, nu = finite_key_output_length(n, k, Q)
out_len_asymptotic = asymptotic_key_length(n, Q)
out_len_placeholder = ToeplitzHashing.calculate_length(
    extractor_type="quantum", input_length=n,
    relative_source_entropy=0.5, error_bound=1e-6,
)

print(f"n = {n} bits, k = {k} (PE sample), Q (from PE sample) = {Q:.4f}")
print(f"Cascade leakage r = {r:.1f} bits, verification hash t = {t:.1f} bits, nu = {nu:.6f}")
print()
print(f"Asymptotic (Shor-Preskill):   {out_len_asymptotic} bits")
print(f"Placeholder (0.5 entropy):    {out_len_placeholder} bits")
print(f"Finite-key corrected:         {out_len_finite} bits")


n = 3488 bits, k = 388 (PE sample), Q (from PE sample) = 0.0077
Cascade leakage r = 286.9 bits, verification hash t = 11.8 bits, nu = 0.261518

Asymptotic (Shor-Preskill):   3032 bits
Placeholder (0.5 entropy):    1706 bits
Finite-key corrected:         186 bits


In [5]:
"""
Apply the validated finite-key formula to your real, collected key pairs,
using each pair's actual PE-sample-derived (k, qber) from
key_pairs_metadata.csv -- NOT a QBER recomputed by comparing the
generation bits directly, which would double-dip on bits that are
supposed to remain unmeasured outside the disjoint PE sample.
"""
import pandas as pd
from qne.cascade.finite_key import finite_key_output_length, asymptotic_key_length
from randextract import ToeplitzHashing

rows = []
for _, row in key_pairs_df.iterrows():
    n_i, k_i, Q_i = int(row["n_bits"]), int(row["k_pe"]), float(row["qber"])

    ell_finite, r, t, nu = finite_key_output_length(n_i, k_i, Q_i)
    ell_asymptotic = asymptotic_key_length(n_i, Q_i)
    ell_placeholder = ToeplitzHashing.calculate_length(
        extractor_type="quantum", input_length=n_i,
        relative_source_entropy=0.5, error_bound=1e-6,
    )

    reduction_pct = (1 - ell_finite / ell_asymptotic) * 100 if ell_asymptotic > 0 else float('nan')

    rows.append({
        "key_index": row["index"], "n_bits": n_i, "k_pe": k_i, "qber": Q_i,
        "cascade_leakage_r": r, "verification_hash_t": t, "nu": nu,
        "ell_asymptotic": ell_asymptotic,
        "ell_placeholder": ell_placeholder,
        "ell_finite_key": ell_finite,
        "reduction_vs_asymptotic_pct": reduction_pct,
    })
    print(f"key{row['index']}: n={n_i}, k={k_i}, Q={Q_i:.4f} -> "
          f"asymptotic={ell_asymptotic}, placeholder={ell_placeholder}, "
          f"finite-key={ell_finite} ({reduction_pct:.1f}% reduction)")

df_finitekey = pd.DataFrame(rows)
print("\n=== Summary across all real keys ===")
print(df_finitekey.to_string(index=False))

df_finitekey.to_csv(str(PROJECT_DIR / "results" / "finite_key_analysis_all_keys.csv"), index=False)
print(f"\nSaved -> {PROJECT_DIR / 'results' / 'finite_key_analysis_all_keys.csv'}")

print(f"\nMean reduction vs. asymptotic: {df_finitekey['reduction_vs_asymptotic_pct'].mean():.1f}%")


key0: n=3448, k=384, Q=0.0130 -> asymptotic=2756, placeholder=1686, finite-key=5 (99.8% reduction)
key1: n=3443, k=383, Q=0.0078 -> asymptotic=2988, placeholder=1683, finite-key=171 (94.3% reduction)
key2: n=3408, k=379, Q=0.0185 -> asymptotic=2503, placeholder=1666, finite-key=1 (100.0% reduction)
key3: n=3395, k=378, Q=0.0185 -> asymptotic=2491, placeholder=1659, finite-key=1 (100.0% reduction)
key4: n=3456, k=384, Q=0.0104 -> asymptotic=2878, placeholder=1690, finite-key=87 (97.0% reduction)

=== Summary across all real keys ===
 key_index  n_bits  k_pe     qber  cascade_leakage_r  verification_hash_t       nu  ell_asymptotic  ell_placeholder  ell_finite_key  reduction_vs_asymptotic_pct
         0    3448   384 0.013021         429.233957            11.751544 0.262918            2756             1686               5                    99.818578
         1    3443   383 0.007833         286.507114            11.749450 0.263234            2988             1683             171         

In [6]:
"""
Compare SDC fault-injection mismatch rates when using the finite-key-
corrected output length vs. the (currently-used) asymptotic/placeholder
output length. Uses key0's real data, Mock session, same fault
probabilities as your earlier sweeps.
"""
import numpy as np
from galois import GF2
from randextract import ToeplitzHashing
from qne.cascade.finite_key import finite_key_output_length, asymptotic_key_length
from qne.cascade.fault_injection import SDCFaultInjector


def run_with_custom_length(alice_key, bob_reconciled, out_len, toeplitz_prob, final_key_prob, seed):
    """Run Toeplitz extraction + fault injection with a MANUALLY SPECIFIED
    output length, instead of the placeholder-based calculate_length call."""
    n_bits = alice_key.get_nr_bits()
    injector = SDCFaultInjector(toeplitz_matrix_prob=toeplitz_prob,
                                  final_key_prob=final_key_prob, seed=seed + 2)

    ext = ToeplitzHashing(input_length=n_bits, output_length=out_len)
    pa_seed = GF2.Random(ext.seed_length)

    alice_final = injector.fast_toeplitz_extract_with_fault(ext, GF2(alice_key.bits), pa_seed, "alice")
    alice_final = injector.maybe_flip_final_key_bit(alice_final, "alice")
    bob_final = injector.fast_toeplitz_extract_with_fault(ext, GF2(bob_reconciled.bits), pa_seed, "bob")
    bob_final = injector.maybe_flip_final_key_bit(bob_final, "bob")

    keys_match = np.array_equal(alice_final, bob_final)
    return keys_match, injector.summary()


# --- Use key0's real, already-reconciled data ---
n = alice_key.get_nr_bits()
k = k_pe
Q = real_qber

ell_finite, _, _, _ = finite_key_output_length(n, k, Q)
ell_asymptotic = asymptotic_key_length(n, Q)
ell_placeholder = ToeplitzHashing.calculate_length(
    extractor_type="quantum", input_length=n, relative_source_entropy=0.5, error_bound=1e-6,
)

print(f"Comparing output lengths: asymptotic={ell_asymptotic}, "
      f"placeholder={ell_placeholder}, finite-key={ell_finite}")

# --- Assume bob_key already reconciles cleanly against alice_key (fault-free Cascade) ---
# If not already reconciled in this session, reconcile first:
from qne.cascade import ORIGINAL, Reconciliation, MockClassicalSession
session = MockClassicalSession(correct_key=alice_key)
reconciliation = Reconciliation(algorithm=ORIGINAL, classical_session=session,
                                  noisy_key=bob_key, estimated_bit_error_rate=Q, seed=42)
bob_reconciled = reconciliation.reconcile()
assert alice_key.nr_bits_different(bob_reconciled) == 0, "Reconciliation didn't converge cleanly!"

# Fast version of the length-comparison sweep, using
# fast_toeplitz_extract_with_fault (no to_matrix() call at all).
rows = []
for length_type, out_len in [("asymptotic", ell_asymptotic),
                               ("placeholder", ell_placeholder),
                               ("finite_key", ell_finite)]:
    ext = ToeplitzHashing(input_length=alice_key.get_nr_bits(), output_length=out_len)

    for prob in [0.1, 0.3, 0.5]:
        for run in range(10):
            seed = 42 + run
            pa_seed = GF2.Random(ext.seed_length)
            injector = SDCFaultInjector(toeplitz_matrix_prob=prob, seed=seed + 2)

            alice_final = injector.fast_toeplitz_extract_with_fault(
                ext, GF2(alice_key.bits), pa_seed, "alice")
            bob_final = injector.fast_toeplitz_extract_with_fault(
                ext, GF2(bob_reconciled.bits), pa_seed, "bob")

            keys_match = np.array_equal(alice_final, bob_final)
            rows.append({
                "length_type": length_type, "out_len": out_len, "toeplitz_prob": prob,
                "run": run, "keys_match": keys_match, "faults_fired": injector.summary(),
            })

df_lengthcompare = pd.DataFrame(rows)
df_lengthcompare["mismatch"] = ~df_lengthcompare["keys_match"]
df_lengthcompare.to_csv(str(PROJECT_DIR / "results" / "sdc_lengthtype_comparison.csv"), index=False)

print(df_lengthcompare[["length_type", "toeplitz_prob", "run", "faults_fired", "mismatch"]].to_string())


Comparing output lengths: asymptotic=3032, placeholder=1706, finite-key=186
    length_type  toeplitz_prob  run            faults_fired  mismatch
0    asymptotic            0.1    0                      {}     False
1    asymptotic            0.1    1                      {}     False
2    asymptotic            0.1    2  {'toeplitz_matrix': 1}     False
3    asymptotic            0.1    3                      {}     False
4    asymptotic            0.1    4                      {}     False
5    asymptotic            0.1    5                      {}     False
6    asymptotic            0.1    6                      {}     False
7    asymptotic            0.1    7                      {}     False
8    asymptotic            0.1    8                      {}     False
9    asymptotic            0.1    9  {'toeplitz_matrix': 1}      True
10   asymptotic            0.3    0  {'toeplitz_matrix': 1}     False
11   asymptotic            0.3    1                      {}     False
12   asymptoti

In [7]:
"""
Side-by-side comparison of mismatch rate across the three output-length
types, with a statistical test for whether any real difference exists.
"""
import pandas as pd
from scipy import stats

df_lengthcompare = pd.read_csv(str(PROJECT_DIR / "results" / "sdc_lengthtype_comparison.csv"))

# --- Side-by-side summary table ---
summary = df_lengthcompare.groupby(["length_type", "toeplitz_prob"])["mismatch"].agg(
    ['sum', 'count', 'mean']).rename(columns={'sum': 'n_mismatch', 'count': 'n_total', 'mean': 'mismatch_rate'})
print("=== Mismatch rate by length type and probability ===")
print(summary.to_string())

# Pivot for a clean side-by-side view
pivot = df_lengthcompare.pivot_table(index="toeplitz_prob", columns="length_type",
                                        values="mismatch", aggfunc="mean")
print("\n=== Side-by-side (rows=probability, columns=length type) ===")
print(pivot.to_string())


# ============================================================
# Statistical test: is there a real difference between length types?
# ============================================================
# Since faults_fired is identical across length types for matching
# (prob, run) pairs (confirmed from your data -- same seed => same
# firing pattern), a paired test is appropriate here, not an
# independent-samples test.

print("\n=== McNemar's test: paired comparison of mismatch outcomes ===")
# McNemar's test is the correct choice for paired binary outcomes
# (same trials, same seeds, just different length -- not independent samples)

for prob in df_lengthcompare["toeplitz_prob"].unique():
    sub = df_lengthcompare[df_lengthcompare["toeplitz_prob"] == prob]

    asym = sub[sub["length_type"] == "asymptotic"].sort_values("run")["mismatch"].values
    fkey = sub[sub["length_type"] == "finite_key"].sort_values("run")["mismatch"].values
    plch = sub[sub["length_type"] == "placeholder"].sort_values("run")["mismatch"].values

    # Build 2x2 contingency table: asymptotic vs finite_key
    both_true = ((asym == True) & (fkey == True)).sum()
    only_asym = ((asym == True) & (fkey == False)).sum()
    only_fkey = ((asym == False) & (fkey == True)).sum()
    both_false = ((asym == False) & (fkey == False)).sum()

    table = [[both_true, only_asym], [only_fkey, both_false]]

    # McNemar's test focuses on the discordant pairs (only_asym, only_fkey)
    n_discordant = only_asym + only_fkey
    if n_discordant == 0:
        print(f"prob={prob}: no discordant pairs (asymptotic and finite_key agree on every trial) -- no difference detected")
        continue

    # scipy doesn't expose a direct McNemar's test; use the classic
    # binomial-based version on the discordant pairs instead.
    from scipy.stats import binomtest
    p_value = binomtest(min(only_asym, only_fkey), n_discordant, 0.5).pvalue if n_discordant > 0 else 1.0

    print(f"prob={prob}: discordant pairs = {n_discordant} (asym-only={only_asym}, fkey-only={only_fkey}), "
          f"McNemar p-value = {p_value:.4f}")

=== Mismatch rate by length type and probability ===
                           n_mismatch  n_total  mismatch_rate
length_type toeplitz_prob                                    
asymptotic  0.1                     1       10            0.1
            0.3                     1       10            0.1
            0.5                     3       10            0.3
finite_key  0.1                     1       10            0.1
            0.3                     1       10            0.1
            0.5                     3       10            0.3
placeholder 0.1                     1       10            0.1
            0.3                     1       10            0.1
            0.5                     3       10            0.3

=== Side-by-side (rows=probability, columns=length type) ===
length_type    asymptotic  finite_key  placeholder
toeplitz_prob                                     
0.1                   0.1         0.1          0.1
0.3                   0.1         0.1          0.1

In [8]:
"""
Extend the length-type comparison to final-key faults (should be
trivially unaffected by output length) and reconciliation-state faults
(structurally can't be affected, since reconciliation happens before
the length choice is applied) -- for completeness and explicit
empirical confirmation.
"""
import pandas as pd
import numpy as np

# ============================================================
# Final-key faults across the three length types
# ============================================================
rows_finalkey = []
for length_type, out_len in [("asymptotic", ell_asymptotic),
                               ("placeholder", ell_placeholder),
                               ("finite_key", ell_finite)]:
    ext = ToeplitzHashing(input_length=alice_key.get_nr_bits(), output_length=out_len)

    for prob in [0.1, 0.3, 0.5]:
        for run in range(10):
            seed = 42 + run
            pa_seed = GF2.Random(ext.seed_length)
            injector = SDCFaultInjector(final_key_prob=prob, seed=seed + 2)

            alice_final = np.array(ext.extract(GF2(alice_key.bits), pa_seed)).copy()
            alice_final = injector.maybe_flip_final_key_bit(alice_final, "alice")

            bob_final = np.array(ext.extract(GF2(bob_reconciled.bits), pa_seed)).copy()
            bob_final = injector.maybe_flip_final_key_bit(bob_final, "bob")

            keys_match = np.array_equal(alice_final, bob_final)
            rows_finalkey.append({
                "length_type": length_type, "out_len": out_len, "final_key_prob": prob,
                "run": run, "keys_match": keys_match, "faults_fired": injector.summary(),
            })

df_finalkey_compare = pd.DataFrame(rows_finalkey)
df_finalkey_compare["mismatch"] = ~df_finalkey_compare["keys_match"]
df_finalkey_compare.to_csv(str(PROJECT_DIR / "results" / "sdc_lengthtype_comparison_finalkey.csv"), index=False)

pivot_finalkey = df_finalkey_compare.pivot_table(index="final_key_prob", columns="length_type",
                                            values="mismatch", aggfunc="mean")
print("=== Final-key faults: mismatch rate by length type ===")
print(pivot_finalkey.to_string())


# ============================================================
# Reconciliation-state faults across the three length types
# (reconciliation runs BEFORE the length choice applies -- testing
# this confirms the length choice has zero effect, as expected)
# ============================================================
from qne.cascade import ORIGINAL, Reconciliation, MockClassicalSession

rows_recon = []
for length_type, out_len in [("asymptotic", ell_asymptotic),
                               ("placeholder", ell_placeholder),
                               ("finite_key", ell_finite)]:
    for prob in [0.03, 0.1, 0.3]:
        for run in range(10):
            seed = 42 + run
            injector = SDCFaultInjector(reconciliation_state_prob=prob, seed=seed + 2)

            session = MockClassicalSession(correct_key=alice_key)
            reconciliation = Reconciliation(
                algorithm=ORIGINAL, classical_session=session, noisy_key=bob_key,
                estimated_bit_error_rate=real_qber, seed=seed, fault_injector=injector,
            )
            try:
                bob_reconciled_test = reconciliation.reconcile()
                non_convergent = False
                remaining = alice_key.nr_bits_different(bob_reconciled_test)
            except RuntimeError:
                non_convergent = True
                remaining = None

            rows_recon.append({
                "length_type": length_type, "out_len": out_len, "reconciliation_prob": prob,
                "run": run, "non_convergent": non_convergent, "remaining_errors": remaining,
                "faults_fired": injector.summary(),
            })

df_recon_compare = pd.DataFrame(rows_recon)
df_recon_compare.to_csv(str(PROJECT_DIR / "results" / "sdc_lengthtype_comparison_reconciliation.csv"), index=False)

pivot_recon = df_recon_compare.pivot_table(index="reconciliation_prob", columns="length_type",
                                              values="non_convergent", aggfunc="mean")
print("\n=== Reconciliation faults: non-convergence rate by length type ===")
print(pivot_recon.to_string())

# Explicit check: are outcomes literally identical across length types (as predicted)?
identical_finalkey = df_finalkey_compare.pivot_table(index=["final_key_prob", "run"], columns="length_type",
                                                values="mismatch").nunique(axis=1).eq(1).all()
identical_recon = df_recon_compare.pivot_table(index=["reconciliation_prob", "run"], columns="length_type",
                                                  values="non_convergent").nunique(axis=1).eq(1).all()
print(f"\nFinal-key outcomes identical across all length types: {identical_finalkey}")
print(f"Reconciliation outcomes identical across all length types: {identical_recon}")


=== Final-key faults: mismatch rate by length type ===
length_type     asymptotic  finite_key  placeholder
final_key_prob                                     
0.1                    0.2         0.2          0.2
0.3                    0.3         0.3          0.3
0.5                    0.6         0.6          0.6

=== Reconciliation faults: non-convergence rate by length type ===
length_type          asymptotic  finite_key  placeholder
reconciliation_prob                                     
0.03                        0.0         0.0          0.0
0.10                        0.0         0.0          0.0
0.30                        0.0         0.0          0.0

Final-key outcomes identical across all length types: True
Reconciliation outcomes identical across all length types: True


In [9]:
df_fk = pd.read_csv(str(PROJECT_DIR / "results" / "finite_key_analysis_all_keys.csv"))
df_fk["gap_pct"] = df_fk["reduction_vs_asymptotic_pct"]
print(df_fk[["key_index", "qber", "gap_pct"]].sort_values("qber"))

   key_index      qber    gap_pct
1          1  0.007833  94.277108
4          4  0.010417  96.977067
0          0  0.013021  99.818578
2          2  0.018470  99.960048
3          3  0.018519  99.959855


In [10]:
df_fk = pd.read_csv(str(PROJECT_DIR / "results" / "finite_key_analysis_all_keys.csv"))
print(df_fk[["key_index", "n_bits", "qber", "ell_asymptotic", "ell_finite_key", "reduction_vs_asymptotic_pct"]])

   key_index  n_bits      qber  ell_asymptotic  ell_finite_key  \
0          0    3448  0.013021            2756               5   
1          1    3443  0.007833            2988             171   
2          2    3408  0.018470            2503               1   
3          3    3395  0.018519            2491               1   
4          4    3456  0.010417            2878              87   

   reduction_vs_asymptotic_pct  
0                    99.818578  
1                    94.277108  
2                    99.960048  
3                    99.959855  
4                    96.977067  


In [11]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

def check(label, condition, detail=""):
    status = "OK" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition

print("=" * 60)
print("KEY0 METADATA CONSISTENCY (cell 1)")
print("=" * 60)

META_PATH = PROJECT_DIR / "results" / "fabric_key0_pe_meta.json"
if META_PATH.exists():
    meta = json.loads(META_PATH.read_text())
    print(f"k={meta['k']}, n={meta['n']}, m={meta['m']}, qber={meta['qber']:.4f}")
    check("k_pe/real_qber in this session match the saved meta file",
          k_pe == meta["k"] and abs(real_qber - meta["qber"]) < 1e-9,
          f"session k={k_pe}, real_qber={real_qber} vs file k={meta['k']}, qber={meta['qber']}")
else:
    print("  fabric_key0_pe_meta.json not found -- run 10_sdc_real.ipynb first.")


print()
print("=" * 60)
print("SINGLE-KEY FINITE-KEY OUTPUT (cell 3)")
print("=" * 60)

check("finite-key length is positive", out_len_finite > 0, f"got {out_len_finite}")
check("finite-key length < asymptotic length (expected: finite-key overhead reduces rate)",
      out_len_finite < out_len_asymptotic,
      f"finite={out_len_finite}, asymptotic={out_len_asymptotic}")
check("cascade leakage r is positive and less than n", 0 < r < n, f"r={r:.1f}, n={n}")
check("verification hash length t is small relative to n", 0 < t < n * 0.1,
      f"t={t:.1f}, n={n}")
check("nu is in the valid range (0, 0.5 - Q)", 0 < nu < (0.5 - Q), f"nu={nu}, 0.5-Q={0.5-Q}")


print()
print("=" * 60)
print("ALL-KEYS FINITE-KEY TABLE (cell 4: finite_key_analysis_all_keys.csv)")
print("=" * 60)

fk_path = PROJECT_DIR / "results" / "finite_key_analysis_all_keys.csv"
if fk_path.exists():
    df_fk = pd.read_csv(str(fk_path))
    print(df_fk.to_string(index=False))

    check("5 rows present", len(df_fk) == 5, f"got {len(df_fk)}")
    check("every ell_finite_key < ell_asymptotic",
          (df_fk["ell_finite_key"] < df_fk["ell_asymptotic"]).all(),
          f"violations: {df_fk[df_fk['ell_finite_key'] >= df_fk['ell_asymptotic']]['key_index'].tolist()}")
    check("reduction_vs_asymptotic_pct all in [0, 100]",
          df_fk["reduction_vs_asymptotic_pct"].between(0, 100).all())
    check("no negative or zero key lengths anywhere",
          (df_fk[["ell_asymptotic", "ell_placeholder", "ell_finite_key"]] > 0).all().all())

    # Cross-check against key_pairs_metadata.csv directly -- these two files
    # should describe the exact same (n, k, qber) per key index.
    meta_path = PROJECT_DIR / "results" / "key_pairs_metadata.csv"
    if meta_path.exists():
        kdf = pd.read_csv(str(meta_path))
        merged = df_fk.merge(kdf, left_on="key_index", right_on="index", suffixes=("_fk", "_meta"))
        check("n_bits matches between finite_key table and key_pairs_metadata",
              (merged["n_bits_fk"] == merged["n_bits_meta"]).all())
        check("k_pe matches between finite_key table and key_pairs_metadata",
              (merged["k_pe_fk"] == merged["k_pe_meta"]).all())
        check("qber matches between finite_key table and key_pairs_metadata",
              np.allclose(merged["qber_fk"], merged["qber_meta"]))
else:
    print("  finite_key_analysis_all_keys.csv not found -- run cell 4 first.")


print()
print("=" * 60)
print("LENGTH-TYPE COMPARISON: TOEPLITZ (cell 5)")
print("=" * 60)

lt_path = PROJECT_DIR / "results" / "sdc_lengthtype_comparison.csv"
if lt_path.exists():
    df_lt = pd.read_csv(str(lt_path))
    check("3 length types present", set(df_lt["length_type"].unique()) == {"asymptotic", "placeholder", "finite_key"})

    # Mismatch rate for a given toeplitz_prob should be roughly SIMILAR across
    # length types -- the fault mechanism doesn't structurally depend on
    # output length (a matrix-entry flip always changes exactly one output
    # bit regardless of how long the output is). This is a soft check; the
    # notebook's own McNemar's test (a later cell) is the rigorous version.
    pivot = df_lt.pivot_table(index="toeplitz_prob", columns="length_type", values="mismatch", aggfunc="mean")
    print(pivot.to_string())
    max_spread = (pivot.max(axis=1) - pivot.min(axis=1)).max()
    check("mismatch rates across length types don't differ wildly for the same prob "
          "(spread < 0.3 -- large spread would suggest length-dependence that "
          "shouldn't exist for this fault mechanism)",
          max_spread < 0.3, f"max spread = {max_spread:.3f}")

    # Higher toeplitz_prob should generally not produce LOWER mismatch rate
    mean_by_prob = df_lt.groupby("toeplitz_prob")["mismatch"].mean().sort_index()
    check("mismatch rate roughly increases with toeplitz_prob (monotonic-ish)",
          mean_by_prob.diff().dropna().ge(-0.15).all(),
          f"{mean_by_prob.to_dict()}")
else:
    print("  sdc_lengthtype_comparison.csv not found -- run cell 5 first.")


print()
print("=" * 60)
print("LENGTH-TYPE COMPARISON: FINAL-KEY + RECONCILIATION (cell 7)")
print("=" * 60)

fk_lt_path = PROJECT_DIR / "results" / "sdc_lengthtype_comparison_finalkey.csv"
recon_lt_path = PROJECT_DIR / "results" / "sdc_lengthtype_comparison_reconciliation.csv"

if fk_lt_path.exists():
    df_fk_lt = pd.read_csv(str(fk_lt_path))
    # Final-key faults: whether a flip FIRES is determined by the RNG's
    # decision draw, which happens before the length-dependent position draw
    # -- so the fire/no-fire outcome (and therefore the mismatch outcome)
    # should be IDENTICAL across length types for a given (prob, run), not
    # just similar. This is a hard check, not a statistical one.
    identical_fk = df_fk_lt.pivot_table(
        index=["final_key_prob", "run"], columns="length_type", values="mismatch"
    ).nunique(axis=1).eq(1).all()
    check("final-key mismatch outcomes are EXACTLY identical across length types "
          "(deterministic RNG-draw-order argument, not just statistically close)",
          identical_fk)
    if not identical_fk:
        disc = df_fk_lt.pivot_table(
            index=["final_key_prob", "run"], columns="length_type", values="mismatch"
        )
        mismatched_rows = disc[disc.nunique(axis=1) > 1]
        print("  Discordant (prob, run) combinations:")
        print(mismatched_rows.to_string())
else:
    print("  sdc_lengthtype_comparison_finalkey.csv not found -- run cell 7 first.")

if recon_lt_path.exists():
    df_recon_lt = pd.read_csv(str(recon_lt_path))
    # Reconciliation happens entirely before the length choice is even
    # computed -- the three length_type loop iterations use the SAME seed
    # per (prob, run), so outcomes should be byte-identical, not just close.
    identical_recon = df_recon_lt.pivot_table(
        index=["reconciliation_prob", "run"], columns="length_type", values="non_convergent"
    ).nunique(axis=1).eq(1).all()
    check("reconciliation non-convergence outcomes are EXACTLY identical across length "
          "types (structurally guaranteed -- reconciliation can't see out_len at all)",
          identical_recon)
    if not identical_recon:
        print("  FAIL DETAIL: this would mean something is leaking state between "
              "length_type iterations (e.g. shared mutable RNG/injector object "
              "reused across iterations) -- investigate before trusting any "
              "reconciliation results in this notebook.")
else:
    print("  sdc_lengthtype_comparison_reconciliation.csv not found -- run cell 7 first.")

print()
print("=" * 60)
print("Sanity check complete.")
print("=" * 60)

KEY0 METADATA CONSISTENCY (cell 1)
k=388, n=3488, m=3876, qber=0.0077
  [OK] k_pe/real_qber in this session match the saved meta file

SINGLE-KEY FINITE-KEY OUTPUT (cell 3)
  [OK] finite-key length is positive
  [OK] finite-key length < asymptotic length (expected: finite-key overhead reduces rate)
  [OK] cascade leakage r is positive and less than n
  [OK] verification hash length t is small relative to n
  [OK] nu is in the valid range (0, 0.5 - Q)

ALL-KEYS FINITE-KEY TABLE (cell 4: finite_key_analysis_all_keys.csv)
 key_index  n_bits  k_pe     qber  cascade_leakage_r  verification_hash_t       nu  ell_asymptotic  ell_placeholder  ell_finite_key  reduction_vs_asymptotic_pct
         0    3448   384 0.013021         429.233957            11.751544 0.262918            2756             1686               5                    99.818578
         1    3443   383 0.007833         286.507114            11.749450 0.263234            2988             1683             171                    94